# EML-GA²M Demo

Quick walkthrough: fit an interpretable symbolic model, get a closed-form formula, and see it extrapolate where XGBoost/EBM fail.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. Generate data: exponential decay

Train on `t in [0, 3]`, test (extrapolate) on `t in [3, 8]`.

In [ ]:
rng = np.random.default_rng(42)
t_train = rng.uniform(0.0, 3.0, 500)
y_train = 5.0 * np.exp(-0.4 * t_train) + rng.normal(0, 0.05, 500)

t_test = np.linspace(0.0, 8.0, 300)
y_true = 5.0 * np.exp(-0.4 * t_test)

print(f"Train: {len(t_train)} points in [0, 3]")
print(f"Test:  {len(t_test)} points in [0, 8] (extrapolation beyond 3)")

## 2. Fit EML-GA²M

In [ ]:
from eml_gam import EMLGAM, TrainConfig

model = EMLGAM(n_features=1, univariate_depth=2, feature_names=["t"])
model.fit(
    t_train.reshape(-1, 1), y_train,
    cfg=TrainConfig(n_epochs=1500),
    warm_start=True,
)

formula = model.total_formula(simplify=True)
print(f"Recovered formula: {formula}")
print(f"Ground truth:      5.0 * exp(-0.4 * t)")

## 3. Compare with XGBoost

XGBoost is a strong interpolator but structurally cannot extrapolate — it predicts a flat constant outside its training range.

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

xgb = XGBRegressor(n_estimators=300, max_depth=6, random_state=42)
xgb.fit(t_train.reshape(-1, 1), y_train)

y_eml = model.predict(t_test.reshape(-1, 1))
y_xgb = xgb.predict(t_test.reshape(-1, 1))

# score on extrapolation region only (t > 3)
mask = t_test > 3.0
r2_eml = r2_score(y_true[mask], y_eml[mask])
r2_xgb = r2_score(y_true[mask], y_xgb[mask])
print(f"Extrapolation R² — EML-GA²M: {r2_eml:+.4f}")
print(f"Extrapolation R² — XGBoost:  {r2_xgb:+.4f}")

## 4. Plot it

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(t_train, y_train, alpha=0.2, s=10, color="gray", label="training data")
ax.plot(t_test, y_true, "k--", lw=1.5, label="ground truth")
ax.plot(t_test, y_eml, "C0-", lw=2, label=f"EML-GA²M (R²={r2_eml:+.2f})")
ax.plot(t_test, y_xgb, "C1-", lw=2, label=f"XGBoost (R²={r2_xgb:+.2f})")
ax.axvline(3.0, ls=":", color="red", alpha=0.5, label="train boundary")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Extrapolation: EML-GA²M recovers exp(-t), XGBoost flattens")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. UCI Yacht Hydrodynamics

Real data: train on slow regime (Fr < 0.3), test on planing regime (Fr >= 0.3).  
Run `python -m scripts.download_datasets` first if you haven't already.

In [ ]:
import pandas as pd
import os

yacht_path = os.path.join("data", "yacht.csv")
if not os.path.exists(yacht_path):
    print("Run: python -m scripts.download_datasets")
else:
    df = pd.read_csv(yacht_path)
    Fr = df["froude_number"].values
    y_all = df["residuary_resistance"].values

    tr_mask = Fr < 0.3
    te_mask = Fr >= 0.3

    # EML-GA²M on Froude number only
    torch.manual_seed(0)
    m = EMLGAM(n_features=1, univariate_depth=2, feature_names=["Fr"])
    m.fit(Fr[tr_mask].reshape(-1, 1), y_all[tr_mask], cfg=TrainConfig(n_epochs=1500))

    y_pred = m.predict(Fr[te_mask].reshape(-1, 1))
    r2 = r2_score(y_all[te_mask], y_pred)
    print(f"Yacht extrapolation R² = {r2:+.4f}")
    print(f"Formula: {m.total_formula(simplify=True)}")

    # XGBoost comparison
    xgb2 = XGBRegressor(n_estimators=300, max_depth=6, random_state=0)
    xgb2.fit(Fr[tr_mask].reshape(-1, 1), y_all[tr_mask])
    y_xgb2 = xgb2.predict(Fr[te_mask].reshape(-1, 1))
    r2_xgb2 = r2_score(y_all[te_mask], y_xgb2)
    print(f"XGBoost extrapolation R² = {r2_xgb2:+.4f}")

    # plot
    order = np.argsort(Fr)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(Fr[tr_mask], y_all[tr_mask], s=15, alpha=0.5, label="train (Fr<0.3)")
    ax.scatter(Fr[te_mask], y_all[te_mask], s=15, alpha=0.5, label="test (Fr>=0.3)")
    Fr_sorted = np.linspace(Fr.min(), Fr.max(), 200)
    ax.plot(Fr_sorted, m.predict(Fr_sorted.reshape(-1, 1)), "C2-", lw=2, label="EML-GA²M")
    ax.plot(Fr_sorted, xgb2.predict(Fr_sorted.reshape(-1, 1)), "C1--", lw=2, label="XGBoost")
    ax.axvline(0.3, ls=":", color="red", alpha=0.5)
    ax.set_xlabel("Froude number")
    ax.set_ylabel("Residuary resistance")
    ax.set_title("UCI Yacht: EML-GA²M extrapolates, XGBoost flattens")
    ax.legend()
    plt.tight_layout()
    plt.show()